# ZAIC 2026 LifeMate — v8-ASR: Whisper large-v3 thay cho large-v3-turbo

ASR vẫn làm mất 12–14 điểm (đo trên v11: transcript chuẩn 52.66 / 69.93, qua ASR v4 40.90 / 55.68). Bản turbo chỉ có
4 lớp decoder; large-v3 có 32 lớp (1.55B). Tổng hệ thống: 4.02B + 1.55B = **5.57B < 6B**, LLM vẫn < 5B.

Notebook này **chỉ đổi ASR**, LLM giữ nguyên adapter **v11** (`ft_v11_adapter`), nên chênh lệch điểm là do ASR.
Mốc của v11 + ASR turbo v4: **WER 11.47%**, val **40.90 / 55.68**, public **49.53**. Thắng khi WER < ~10.5% **và** public > 50.5.

- `HOLD_OUT=True` (lần chạy đầu): giữ clip val ra ngoài để đo WER. Public dùng bản này luôn.
- `HOLD_OUT=False` (lần chạy cuối, khi đã biết base nào thắng): train trên toàn bộ 5036 clip, bỏ qua phần đo trên val.

**GPU: A100** (large-v3 cần khoảng 30GB khi train; L4 24GB không đủ). Khoảng 1.5–2 giờ.

In [ ]:
# ==== CẤU HÌNH ====
ASR_BASE   = "openai/whisper-large-v3"        # hoặc "openai/whisper-large-v3-turbo" (công thức v4)
HOLD_OUT   = True       # True: giữ clip val để đo. False: train trên toàn bộ clip (lần cuối)
ASR_EPOCHS = 3
BATCH      = 32
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
DATA_URL   = "https://dl-challenge.zalo.ai/2026/lifemate_argfzdgert/train+public_test.zip"
DRIVE_DIR  = "/content/drive/MyDrive/ZAIC"
ADAPTER    = DRIVE_DIR + "/ft_v11_adapter"      # LLM v11 (public 49.53) — giữ cố định để so sánh sạch
TAG        = ("large" if ASR_BASE.endswith("large-v3") else "turbo") + ("_holdout" if HOLD_OUT else "_all")
WORK, DATA_RAW, RES = "/content/work", "/content/data_raw", f"/content/results_v8asr_{TAG}"
LLM, ASR_FT = "/content/ft_fit/merged", "/content/asr_ft"
ASR_DRIVE  = f"{DRIVE_DIR}/asr_v8_{TAG}"
print("TAG:", TAG, "->", ASR_DRIVE)

In [ ]:
# ==== KIỂM TRA ĐÚNG FILE (chạy đầu tiên) ====
NOTEBOOK = "v8_asr_large_v3"
from google.colab import drive; drive.mount("/content/drive")
import os, zipfile
_z = os.path.join(DRIVE_DIR, "lifemate_baseline.zip")
assert os.path.exists(_z), "Drive không có MyDrive/ZAIC/lifemate_baseline.zip"
_v = zipfile.ZipFile(_z).read("VERSION").decode().strip() if "VERSION" in zipfile.ZipFile(_z).namelist() else "(không có VERSION — zip cũ)"
assert _v == "2026-11-13-1200", f"lifemate_baseline.zip trên Drive là bản {_v}, cần bản 2026-11-13-1200 → ghi đè bằng bản mới nhất (zip của repo)"
_need = ["ft_v11_adapter/adapter_config.json", "asr_ft_v4/config.json"]
_miss = [p for p in _need if not os.path.exists(os.path.join(DRIVE_DIR, p))]
assert not _miss, "Thiếu trên Drive: " + ", ".join(_miss)
print(f"OK: notebook {NOTEBOOK} | zip {_v} | đủ: {_need}")


In [ ]:
import subprocess, torch, os
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Chưa bật GPU"
DTYPE = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"
GPU_NAME = torch.cuda.get_device_name(0); print(GPU_NAME, DTYPE)
os.makedirs(RES, exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile, shutil, glob, subprocess, json, time
shutil.rmtree(WORK, ignore_errors=True); zipfile.ZipFile(os.path.join(DRIVE_DIR, "lifemate_baseline.zip")).extractall(WORK)
assert os.path.exists(f"{WORK}/scripts/detcheck.py"), "lifemate_baseline.zip trong Drive là bản cũ -> upload bản mới"
data_zip = os.path.join(DRIVE_DIR, "train+public_test.zip")
if not os.path.exists(data_zip):
    data_zip = "/content/train+public_test.zip"
    if not os.path.exists(data_zip):
        assert subprocess.run(["wget", "-q", "-O", data_zip, DATA_URL]).returncode == 0, "Tải dữ liệu lỗi"
if not glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True):
    zipfile.ZipFile(data_zip).extractall(DATA_RAW)
DATA = os.path.dirname(os.path.dirname(glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True)[0]))
print("DATA:", DATA)
assert os.path.exists(ADAPTER + "/adapter_config.json"), "Không thấy adapter v11 trong Drive: " + ADAPTER

In [ ]:
%%capture pipout
!pip install -q vllm peft soundfile
!pip uninstall -y torchao

In [ ]:
print(pipout.stderr[-800:])
!pip list 2>/dev/null | grep -iE "^(vllm|torch|transformers|peft) " | tee {RES}/versions.txt
%cd {WORK}
!cat splits/val_unseen.txt splits/val_seen.txt > splits/val_all.txt

## 1. Fine-tune Whisper. Theo dõi `loss` / `eta`

In [ ]:
%cd {WORK}
EXCL = "--exclude splits/val_all.txt" if HOLD_OUT else ""
!python -m lifemate.asr_finetune --root {DATA} {EXCL} --model {ASR_BASE} --out {ASR_FT} --epochs {ASR_EPOCHS} 2>&1 | grep -vi warn
assert os.path.exists(f"{ASR_FT}/config.json"), "ASR fine-tune lỗi: xem log phía trên"
shutil.copy(f"{ASR_FT}/asr_train_log.txt", f"{RES}/asr_train_log.txt")
shutil.copytree(ASR_FT, ASR_DRIVE, dirs_exist_ok=True); print("ASR saved to Drive:", ASR_DRIVE)

## 2. WER trên clip val (mốc turbo v4: **11.47%**) — chỉ khi `HOLD_OUT=True`

In [ ]:
%cd {WORK}
if HOLD_OUT:
    !python -m lifemate.asr --root {DATA} --split train --ids splits/val_all.txt --model {ASR_FT} --dtype {DTYPE} --out asr_val.json 2>&1 | grep -vi warn | tail -4
    shutil.copy("asr_val.json.wer.txt", f"{RES}/wer.txt")
    !head -3 {RES}/wer.txt
else:
    print("HOLD_OUT=False: clip val đã nằm trong tập train, bỏ qua")

## 3. Điểm end-to-end trên val, LLM v11 (mốc với turbo: **40.90 / 55.68**) — chỉ khi `HOLD_OUT=True`

In [ ]:
%cd {WORK}
!python -m lifemate.merge --base {BASE_MODEL} --adapter {ADAPTER} --out {LLM} 2>&1 | grep -vi warn | tail -1
sc = {}
if HOLD_OUT:
    for S in ["val_unseen", "val_seen"]:
        !python -m lifemate.run --root {DATA} --split train --ids splits/{S}.txt --backend vllm --model {LLM} --dtype {DTYPE} --batch {BATCH} --transcripts asr_val.json --out runs/v8_{S} 2>&1 | tail -1
        !python scripts/score.py --root {DATA} --split train --pred runs/v8_{S}/submission.jsonl --by-tag > {RES}/score_{S}.txt
        !python -m lifemate.report --root {DATA} --split train --pred runs/v8_{S}/submission.jsonl --show 6 > {RES}/report_{S}.txt
        sc[S] = float(open(f"{RES}/score_{S}.txt").readline().split()[-1])
    print(TAG, sc, "| v11 + turbo v4: {'val_unseen': 40.9, 'val_seen': 55.68}")
json.dump(sc, open(f"{RES}/e2e.json", "w"))

## 4. Public → `submission_public_v8asr_<TAG>.jsonl` (mốc v11 + turbo: **49.53**)

In [ ]:
%cd {WORK}
t0 = time.time()
!python -m lifemate.asr --root {DATA} --split public_test --model {ASR_FT} --dtype {DTYPE} --out asr_public.json 2>&1 | tail -1
asr_sec = time.time() - t0
!python -m lifemate.run --root {DATA} --split public_test --backend vllm --model {LLM} --dtype {DTYPE} --batch {BATCH} --transcripts asr_public.json --out runs/public 2>&1 | tail -1
SUB = f"submission_public_v8asr_{TAG}.jsonl"
shutil.copy("runs/public/submission.jsonl", f"{RES}/{SUB}")
shutil.copy("runs/public/submission.jsonl", os.path.join(DRIVE_DIR, SUB))
shutil.copy("runs/public/traces.jsonl", f"{RES}/traces_public.jsonl")
shutil.copy("asr_public.json", f"{RES}/asr_public.json")
print(f"ASR public: {asr_sec:.0f}s cho 634 mẫu ({asr_sec / 634:.2f}s/mẫu, batch 16)")
%cd {DATA}/public_test
!python -m zaic_tools --split public_test --root .. --validate {RES}/{SUB} 2>&1 | tail -3 | tee {RES}/validate_public.txt

## 5. Đóng gói

In [ ]:
open(f"{RES}/config.txt", "w").write(f"ASR_BASE={ASR_BASE}\nHOLD_OUT={HOLD_OUT}\nepochs={ASR_EPOCHS}\nLLM=ft_v11_adapter (v11)\nGPU={GPU_NAME}\nasr_public_sec={asr_sec:.0f}\n")
z = shutil.make_archive(f"/content/results_v8asr_{TAG}", "zip", RES)
shutil.copy(z, os.path.join(DRIVE_DIR, os.path.basename(z))); print("saved", os.path.basename(z))
from google.colab import files
files.download(z); files.download(f"{RES}/{SUB}")